<a href="https://colab.research.google.com/github/matheusmanfron/Projeto-I.A-2.0/blob/main/ScriptsPython/Tratamento_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Importando a biblioteca pandas e definindo meu DataFrame
import pandas as pd

df = pd.read_csv("Partidas_Compactadas.csv")

In [2]:
#Padronização de tipos dos dados
df["data"] = pd.to_datetime(
  df["data"],
  errors="coerce",
)

colunas_inteiras = [
    "fixture_id",
    "team_id",
    "adversario_id",
    "resultado",
    "league_id",
    "season_id",
    "state_id",
]

for coluna in colunas_inteiras:
  df[coluna] = pd.to_numeric(
      df[coluna],
      errors="coerce",
  ).astype("Int64")

  colunas_contagem = [
      "Escanteios",
      "Gols",
      "Assistências",
      "Cartões vermelhos",
      "Cartões amarelos",
  ]

  for coluna in colunas_contagem:
    df[coluna] = pd.to_numeric(
        df[coluna],
        errors="coerce",
    ).astype("Int64")

    colunas_percentuais = [
        "Posse de bola (%)",
        "Dribles bem-sucedidos (%)",
    ]

    for coluna in colunas_percentuais:
      df[coluna] = pd.to_numeric(
          df[coluna],
          errors="coerce"
      )

      print(df.dtypes)

fixture_id                            Int64
data                         datetime64[ns]
partida                              object
time                                 object
team_id                               int64
local_visitante                      object
adversario                           object
adversario_id                         int64
resultado                             int64
league_id                             int64
season_id                             int64
state_id                              int64
Escanteios                            Int64
Posse de bola (%)                   float64
Gols                                float64
Assistências                        float64
Cartões vermelhos                   float64
Cartões amarelos                    float64
Dribles bem-sucedidos (%)           float64
dtype: object
fixture_id                            Int64
data                         datetime64[ns]
partida                              object
time              

In [3]:
#Padronização dos dados tipos texto (string) e nomes das colunas
colunas_texto = [
    "partida",
    "time",
    "local_visitante",
    "adversario",
]

for colunas in colunas_texto:
  df[coluna] = (
      df[coluna]
      .astype("string")
      .str.strip()
  )

  df["local_visitante"] = (
      df["local_visitante"]
      .str.lower()
  )


  df["time"] = df["time"].astype("string")
  print(df["time"].dtype)

string
string
string
string


In [4]:
#Conferindo colunas nulas
df = df.copy()

colunas_estatisticas = [
    "Escanteios",
    "Posse de bola (%)",
    "Gols",
    "Assistências",
    "Cartões vermelhos",
    "Cartões amarelos",
    "Dribles bem-sucedidos (%)"
]

relatorio_nulos = pd.DataFrame(
    {
        "quantidade": df[
            colunas_estatisticas
        ].isna().sum(),

        "percentual": (
            df[colunas_estatisticas]
            .isna()
            .mean()
            .mul(100)
            .round(2)
        ),
    }
)

print(relatorio_nulos)

                           quantidade  percentual
Escanteios                          5        0.13
Posse de bola (%)                  67        1.79
Gols                              206        5.51
Assistências                     1981       53.01
Cartões vermelhos                 885       23.68
Cartões amarelos                  190        5.08
Dribles bem-sucedidos (%)        1219       32.62


In [6]:
#Tratando valores nulos

ano_inicio = 2017
ano_assitencias = 2021
limite_nulos_coluna = 0.20

log = {}

#conferindo o padrão temporal dos valores nulos
df.assign(ano=df["data"].dt.year).groupby("ano")[colunas_estatisticas].agg(
    lambda s: s.isna().mean().round(2)
)

,Escanteios,Posse de bola (%),Gols,Assistências,Cartões vermelhos,Cartões amarelos,Dribles bem-sucedidos (%)
ano,,,,,,,
2009,1,1.00,0.00,1.00,1.00,1.00,1.00
2010,1,1.00,0.00,1.00,1.00,1.00,1.00
2014,0,0.10,0.86,1.00,1.00,1.00,1.00
2015,0,0.00,0.87,1.00,1.00,1.00,1.00
2016,0,0.04,0.26,1.00,0.28,0.28,1.00
2017,0,0.02,0.00,1.00,0.00,0.00,1.00
2018,0,0.02,0.00,1.00,0.01,0.00,1.00
2019,0,0.12,0.00,1.00,0.00,0.00,0.62
2020,0,0.00,0.00,1.00,0.00,0.00,0.05


In [7]:
#Tratando valores nulos (Removendo período sem coleta)

n_antes = len(df)
df = df[df["data"].dt.year >= ano_inicio].copy().reset_index(drop=True)
log["Linhas removidas (anteriores a 2017)"] = n_antes - len(df)
print(f"{n_antes - len(df)} linhas removidas; restam {len(df)}")

291 linhas removidas; restam 3446


In [8]:
#Tratando valores nulos (Gols)
print("Resultado das linhas com Gols nulos:")
print(df.loc[df["Gols"].isna(), "resultado"].value_counts().rename({0: "derrota", 1: "empate", 2: "vitoria"}))

mask = df["Gols"].isna() & df["resultado"].isin([0, 1]).to_numpy()
df.loc[mask, "Gols"] = 0
log["Gols -> 0 (empate/derrota)"] = int(mask.sum())
log["Gols nulos mantido (vitoria)"] = int(df["Gols"].isna().sum())

Resultado das linhas com Gols nulos:
resultado
empate     51
derrota     9
vitoria     8
Name: count, dtype: Int64


In [10]:
#Tratando valores nulos (cartões vermelhos, dribles e assistencias)

n = df["Cartões vermelhos"].isna().sum()
df["Cartões vermelhos"] = df["Cartões vermelhos"].fillna(0)
log["Cartões vermelhos → 0"] = int(n)

ano = df["data"].dt.year
mask = df["Assistências"].isna() & (ano >= ano_assitencias)
df.loc[mask, "Assistências"] = 0
log[f"Assistências → 0 ({ano_assitencias}+)"] = int(mask.sum())
log[f"Assistências nulas antes de {ano_assitencias} (estruturais)"] = int(df["Assistências"].isna().sum())

#Descartando Assitencias e dribles bem sucedidos
pct = df[colunas_estatisticas].isna().mean()
descartar = pct[pct > limite_nulos_coluna].index.tolist()
df = df.drop(columns=descartar)
log["Colunas descartadas"] = ", ".join(descartar) if descartar else "nenhuma"
print("Descartadas:", descartar)

Descartadas: ['Assistências', 'Dribles bem-sucedidos (%)']


In [11]:
df.to_csv("Partidas_Tratadas.csv", index=False)
print("Base tratada e salva")

Base tratada e salva
